# Weather: Open-Meteo API
Open-Meteo exposes three separate endpoints. This notebook calls each one and records
what it returns.

In [1]:
import time
import pandas as pd
import requests

pd.set_option("display.max_columns", 200)

LAT, LON = 40.64, -73.78  # JFK, test location

VARIABLES = [
    "temperature_2m",
    "precipitation",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "weather_code",
]


def fetch(url, hourly, params_extra=None):
    params = {
        "latitude": LAT,
        "longitude": LON,
        "hourly": ",".join(hourly),
        "models": "gfs_seamless",
        "timezone": "UTC",
        **(params_extra or {}),
    }
    r = requests.get(url, params=params, timeout=60)
    return r.status_code, r.json()


def coverage(payload, requested):
    hourly = payload.get("hourly", {})
    rows = []
    for var in requested:
        values = hourly.get(var, [])
        rows.append({
            "variable": var,
            "returned": len(values),
            "non_null": sum(v is not None for v in values),
        })
    return pd.DataFrame(rows)

## 1. Historical Forecast API

`historical-forecast-api.open-meteo.com/v1/forecast`

Returns the forecast that was issued for a given past date - the most recent one
available for that date.

In [2]:
HISTORICAL_FORECAST = "https://historical-forecast-api.open-meteo.com/v1/forecast"

status, payload = fetch(
    HISTORICAL_FORECAST, VARIABLES, {"start_date": "2025-06-10", "end_date": "2025-06-12"}
)
print(f"status {status}")
coverage(payload, VARIABLES)

status 200


,variable,returned,non_null
0,temperature_2m,72,72
1,precipitation,72,72
2,snowfall,72,72
3,wind_speed_10m,72,72
4,wind_gusts_10m,72,72
5,weather_code,72,72


In [3]:
pd.DataFrame(payload["hourly"]).head()

,time,temperature_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m,weather_code
0,2025-06-10T00:00,18.1,0.0,0.0,4.3,4.7,3
1,2025-06-10T01:00,17.3,0.0,0.0,2.2,2.9,3
2,2025-06-10T02:00,17.1,0.0,0.0,0.8,1.4,3
3,2025-06-10T03:00,17.1,0.0,0.0,2.9,4.7,45
4,2025-06-10T04:00,17.1,0.0,0.0,6.2,7.9,45


In [4]:
# Earliest date that returns data for GFS.
probe_dates = ["2021-01-10", "2023-01-10", "2024-01-10", "2025-01-10"]

rows = []
for date in probe_dates:
    status, payload = fetch(HISTORICAL_FORECAST, VARIABLES, {"start_date": date, "end_date": date})
    cov = coverage(payload, VARIABLES)
    rows.append({"date": date, "status": status, "variables_populated": int((cov["non_null"] > 0).sum())})
    time.sleep(1)

pd.DataFrame(rows)

,date,status,variables_populated
0,2021-01-10,200,6
1,2023-01-10,200,6
2,2024-01-10,200,6
3,2025-01-10,200,6


## 2. Previous Runs API

`previous-runs-api.open-meteo.com/v1/forecast`

Returns the forecast as it stood N days before the target date. Each variable is
requested with a `_previous_dayN` suffix.

In [5]:
PREVIOUS_RUNS = "https://previous-runs-api.open-meteo.com/v1/forecast"


def previous_runs_vars(lead, variables=VARIABLES):
    return [f"{v}_previous_day{lead}" for v in variables]


status, payload = fetch(
    PREVIOUS_RUNS,
    previous_runs_vars(3),
    {"start_date": "2025-06-10", "end_date": "2025-06-12"},
)
print(f"status {status}")
coverage(payload, previous_runs_vars(3))

status 200


,variable,returned,non_null
0,temperature_2m_previous_day3,72,72
1,precipitation_previous_day3,72,72
2,snowfall_previous_day3,72,72
3,wind_speed_10m_previous_day3,72,72
4,wind_gusts_10m_previous_day3,72,72
5,weather_code_previous_day3,72,72


In [6]:
pd.DataFrame(payload["hourly"]).head()

,time,temperature_2m_previous_day3,precipitation_previous_day3,snowfall_previous_day3,wind_speed_10m_previous_day3,wind_gusts_10m_previous_day3,weather_code_previous_day3
0,2025-06-10T00:00,19.1,0.0,0.0,14.5,16.9,0
1,2025-06-10T01:00,18.4,0.0,0.0,13.9,17.6,0
2,2025-06-10T02:00,18.1,0.0,0.0,14.2,20.2,0
3,2025-06-10T03:00,18.0,0.0,0.0,13.5,20.2,1
4,2025-06-10T04:00,18.0,0.0,0.0,11.8,16.6,1


In [7]:
# Which lead times respond, on a recent date.
rows = []
for lead in range(1, 9):
    requested = previous_runs_vars(lead)
    status, payload = fetch(
        PREVIOUS_RUNS, requested, {"start_date": "2026-06-10", "end_date": "2026-06-12"}
    )
    cov = coverage(payload, requested)
    rows.append({
        "lead_days": lead,
        "status": status,
        "variables_populated": int((cov["non_null"] > 0).sum()),
    })
    time.sleep(1)

pd.DataFrame(rows)

,lead_days,status,variables_populated
0,1,200,6
1,2,200,6
2,3,200,6
3,4,200,6
4,5,200,6
5,6,200,6
6,7,200,6
7,8,200,0


In [8]:
# Earliest date with full variable coverage at a fixed lead time.
probe_dates = ["2023-12-20", "2024-06-10", "2024-09-10", "2024-12-10", "2025-06-10"]

rows = []
for date in probe_dates:
    requested = previous_runs_vars(3)
    status, payload = fetch(PREVIOUS_RUNS, requested, {"start_date": date, "end_date": date})
    cov = coverage(payload, requested)
    rows.append({"date": date, "status": status, "variables_populated": int((cov["non_null"] > 0).sum())})
    time.sleep(1)

pd.DataFrame(rows)

,date,status,variables_populated
0,2023-12-20,200,1
1,2024-06-10,200,4
2,2024-09-10,200,4
3,2024-12-10,200,6
4,2025-06-10,200,6


## 3. Forecast API

`api.open-meteo.com/v1/forecast`

Returns the current forecast for today and the days ahead, starting from now. Unlike
the other two, it does not accept a `start_date` in the past.

In [9]:
FORECAST = "https://api.open-meteo.com/v1/forecast"

status, payload = fetch(FORECAST, VARIABLES, {"forecast_days": 7})
print(f"status {status}")
coverage(payload, VARIABLES)

status 200


,variable,returned,non_null
0,temperature_2m,168,168
1,precipitation,168,168
2,snowfall,168,168
3,wind_speed_10m,168,168
4,wind_gusts_10m,168,168
5,weather_code,168,168


In [10]:
df = pd.DataFrame(payload["hourly"])
df["time"] = pd.to_datetime(df["time"])
print(df["time"].min(), "->", df["time"].max())
df.head()

2026-08-08 00:00:00 -> 2026-08-14 23:00:00


,time,temperature_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m,weather_code
0,2026-08-08 00:00:00,22.9,0.4,0.0,2.9,15.8,51
1,2026-08-08 01:00:00,23.1,0.0,0.0,3.2,6.5,3
2,2026-08-08 02:00:00,23.0,0.0,0.0,1.3,2.2,3
3,2026-08-08 03:00:00,23.1,0.0,0.0,3.5,6.1,0
4,2026-08-08 04:00:00,23.0,0.0,0.0,5.7,12.6,1


In [11]:
# Maximum forecast_days accepted.
for days in [7, 16, 20]:
    status, payload = fetch(FORECAST, ["temperature_2m"], {"forecast_days": days})
    print(f"forecast_days={days}: status {status}, reason: {payload.get('reason', '-')}")
    time.sleep(1)

forecast_days=7: status 200, reason: -
forecast_days=16: status 200, reason: -
forecast_days=20: status 400, reason: Forecast days is invalid. Allowed range 0 to 16. Given 16.
